### Rationalisierung – Lösung Schritt für Schritt

*32. Bundeswettbewerb Informatik (2013), 1. Runde, Aufgabe 1*

Obwohl der Lehrer es nicht erlaubt, benutzt Anna bei ihren Mathe-Hausaufgaben immer den Taschenrechner. Als Ergebnis erhält sie dabei häufig eine Fließkommazahl. Damit ist der Lehrer aber nicht zufrieden; er erwartet als Ergebnis einen Bruch, also eine rationale Zahl p/q.

Einmal zeigte der Taschenrechner, nach einer langen Rechnung, die Zahl x = 1,857143 an. Anna gab ihr Ergebnis so an: x = 1857143/1000000. Aber das akzeptierte ihr Lehrer nicht; er sagte, das richtige Ergebnis sei x = 13/7. Da erinnerte sich Anna daran, dass in den meisten Schulaufgaben die Ergebnisbrüche kleine Zähler und Nenner haben.

Für die Zukunft sucht Anna also immer eine rationale Zahl, die „möglichst nahe“ am Taschenrechner-Ergebnis ist und „möglichst kleine“ Zähler und Nenner hat.

#### Aufgabe 1

Schreibe ein Programm, das Anna hilft.

Beschreibe dafür genauer, was mit den Zielsetzungen „möglichst klein“ und „möglichst nahe“ gemeint sein könnte und wie diese verschiedenen Ziele miteinander vereinbart werden können. Was gibt dein Programm für die folgenden Fließkommazahlen aus:

    1,293982
    3,141593
    1,414214

**Zu den Daten:** Beispieldaten gibt es zu dieser Aufgabe nicht. In `eigene_testdaten/zahlen.txt` stehen die vier Zahlen aus der Aufgabe und aus den Lösungshinweisen, jeweils als Text mit Dezimalpunkt.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zwei Ziele, die sich widersprechen:** Je genauer ein Bruch, desto größer meist Zähler und Nenner. Man muss die Ziele erst genau fassen (messen!) und dann entscheiden, wie man sie vereinbart. Hier hilft ein Blick auf alle „Kompromisse“: Brüche, die genauer sind als alle Brüche mit kleinerem Nenner.
- **Durchprobieren (Brute Force):** Für jeden Nenner `q` gibt es genau einen besten Zähler, nämlich den, der `x · q` am nächsten liegt. Man muss also nur die Nenner der Reihe nach durchgehen, nicht auch noch die Zähler.
- **Der euklidische Algorithmus** liefert den größten gemeinsamen Teiler (ggT) und damit das Kürzen.
- **Fehler aus der Eingabe:** Eine Zahl mit sechs Nachkommastellen ist schon gerundet. Genauer als die halbe letzte Stelle kann man also nie verlangen.

**Nützliche Funktionen und Bibliotheken**
- `fractions.Fraction`: rechnet **exakt** mit Brüchen (`Fraction('1.293982')`, `Fraction(13, 7)`), kürzt von selbst und kann vergleichen und subtrahieren. Mit Fließkommazahlen würden Rundungsfehler die Vergleiche verfälschen.
- `round(x)`: auf die nächste ganze Zahl runden (auch bei einem `Fraction`).
- `abs(x)`: Betrag.

## Die Idee in Worten

**Möglichst nahe.** Der Abstand `|x − p/q|` zwischen der Zahl und dem Bruch. Kleiner ist besser.

**Möglichst klein.** Ein Bruch ist einfacher, wenn Zähler oder Nenner weniger Stellen haben. Bei einer festen Zahl `x` ist der Zähler ungefähr `x · q`, hängt also am Nenner. Wir messen die Einfachheit deshalb am **Nenner `q`**: je kleiner `q`, desto einfacher. Der Zähler wird nebenbei mit kleiner.

**Beide Ziele zugleich.** Es gibt keinen Bruch, der in beidem gewinnt. Wir untersuchen deshalb die **Kompromisse**: Ein Bruch ist ein Kompromiss, wenn jeder Bruch mit kleinerem Nenner weniger genau ist. Wer mehr Genauigkeit will als so einen Bruch, zahlt mit einem größeren Nenner. Die Kompromisse bilden eine Kurve von „sehr einfach, ungenau“ bis „genau, aber unhandlich“, und wir müssen eine Stelle auf dieser Kurve wählen. Zwei Möglichkeiten:

1. **Die Stellenzahl vorgeben:** Der Lehrer akzeptiert Nenner mit höchstens `n` Stellen; dann nehmen wir unter allen Brüchen mit höchstens `n` Stellen im Nenner den genauesten.
2. **Die Genauigkeit der Eingabe nutzen** (unsere Hauptlösung): Der Taschenrechner hat die Zahl auf 6 Stellen **gerundet**. Jeder Bruch, der sich von `x` um höchstens eine halbe Einheit der letzten Stelle unterscheidet (hier 0,0000005), könnte ebenso gut die „wahre“ Zahl sein. Unter diesen nehmen wir den mit dem **kleinsten Nenner**. Genauer als die Eingabe zu sein, bringt nichts, und einfacher darf man nicht werden, ohne die Eingabe zu verfälschen.

**Vorgehen.** Zahlen exakt einlesen, Abstand und Einfachheit messen, die Kompromisse berechnen, daraus beide Verfahren ablesen.

## Schritt 1: Zahlen exakt einlesen und kürzen

Die Zahlen stehen als Text mit Dezimalpunkt in der Datei. `Fraction('1.857143')` macht daraus den exakten Bruch 1857143/1000000, ohne Rundungsfehler.

In [ ]:
from fractions import Fraction


def lies_zahlen(dateiname):
    zahlen = []
    for zeile in open(dateiname):
        if zeile.strip():
            zahlen.append(zeile.strip())
    return zahlen


zahlen = lies_zahlen('eigene_testdaten/zahlen.txt')
print(zahlen)
x = Fraction(zahlen[0])
print(x, '=', x.numerator, '/', x.denominator)

['1.857143', '1.293982', '3.141593', '1.414214']
1857143/1000000 = 1857143 / 1000000


`Fraction` kürzt selbst. Damit du siehst, wie das geht, hier der **euklidische Algorithmus**: Man teilt `a` durch `b`; der Rest ersetzt `b`, das alte `b` ersetzt `a`, bis der Rest 0 ist. Dann steht der ggT in `a`. (Die Hinweise ziehen stattdessen immer das Kleinere vom Größeren ab; das ist dasselbe, nur langsamer.)

In [ ]:
def ggt(a, b):
    while b != 0:
        a, b = b, a % b
    return a


def kuerze(zaehler, nenner):
    g = ggt(zaehler, nenner)
    return zaehler // g, nenner // g


print(ggt(1857143, 1000000), kuerze(1857143, 1000000))
print(ggt(39, 21), kuerze(39, 21))
print(kuerze(1293982, 1000000))

1 (1857143, 1000000)
3 (13, 7)
(646991, 500000)


Annas Bruch 1857143/1000000 ist schon gekürzt, und 1293982/1000000 wird nur zu 646991/500000. **Kürzen ist verlustfrei, aber noch lange nicht einfach genug.**

## Schritt 2: Abstand und Einfachheit messen

`abstand` ist der Betrag der Differenz, als exakter `Fraction`. `stellen` zählt die Stellen des Nenners.

In [ ]:
def abstand(x, bruch):
    return abs(x - bruch)


def stellen(bruch):
    return len(str(bruch.denominator))


b = Fraction(13, 7)
print(float(abstand(x, b)), stellen(b))
b = Fraction(1857143, 1000000)
print(float(abstand(x, b)), stellen(b))

1.4285714285714285e-07 1
0.0 7


13/7 ist mit einem Abstand von etwa 0,00000014 und einem einstelligen Nenner **schon sehr gut**, Annas Bruch hat Abstand 0 und einen siebenstelligen Nenner. Der winzige Abstand von 13/7 ist kleiner als die Rundung, die der Taschenrechner selbst vorgenommen hat.

## Schritt 3: Die Kompromisse

Wir gehen alle Nenner `q` von 1 bis zu einer Obergrenze durch. Zu jedem `q` gibt es einen besten Zähler `p`, nämlich `x · q`, auf eine ganze Zahl gerundet (mindestens 1, damit wir nicht bei `0/q` landen). Ist `p/q` **genauer als alles bisher**, ist es ein Kompromiss und kommt in die Liste. `Fraction` kürzt von selbst; ein ungekürzter Bruch wäre nicht genauer als sein gekürzter mit kleinerem Nenner und kommt daher gar nicht erst in die Liste. Je weiter hinten ein Bruch steht, desto genauer, aber auch desto größer der Nenner.

In [ ]:
def kompromisse(x, grenze):
    liste = []
    bester_abstand = None
    for q in range(1, grenze + 1):
        p = max(1, round(x * q))
        bruch = Fraction(p, q)
        if bester_abstand is None or abstand(x, bruch) < bester_abstand:
            bester_abstand = abstand(x, bruch)
            liste.append(bruch)
    return liste


for text in zahlen[1:]:
    liste = kompromisse(Fraction(text), 9999)
    print(text, len(liste), 'Kompromisse:')
    print('   ', ', '.join(str(b) for b in liste))

1.293982 25 Kompromisse:
    1, 3/2, 4/3, 9/7, 13/10, 22/17, 295/228, 317/245, 339/262, 361/279, 383/296, 405/313, 427/330, 449/347, 471/364, 493/381, 515/398, 537/415, 559/432, 3376/2609, 3935/3041, 4494/3473, 5053/3905, 5612/4337, 11783/9106
3.141593 14 Kompromisse:
    3, 13/4, 16/5, 19/6, 22/7, 179/57, 201/64, 223/71, 245/78, 267/85, 289/92, 311/99, 333/106, 355/113
1.414214 17 Kompromisse:
    1, 3/2, 4/3, 7/5, 17/12, 24/17, 41/29, 58/41, 99/70, 239/169, 338/239, 577/408, 1393/985, 1970/1393, 8457/5980, 10427/7373, 12397/8766


So liest man die Kurve: Für 3,141593 ist 22/7 ein guter Kompromiss, denn der nächste, 179/57, ist kaum genauer. Erst 355/113 bringt einen großen Sprung, und danach gibt es bis Nenner 9999 **gar keinen** besseren Bruch mehr. Bei 1,414214 sind 99/70, 1393/985 und 1970/1393 Stufen mit jeweils einem großen Sprung in der Genauigkeit. Bei 1,293982 ist die Kurve unregelmäßig: Zwischen 22/17 und 559/432 liegen 12 Kompromisse, die sich nur kleinschrittig verbessern.

**Die beste Näherung mit höchstens `n` Nennerstellen** ist der letzte Kompromiss mit Nenner unter `10^n`. Hier die Tabelle für n = 2, 3, 4 (der Abstand ist `|x − p/q|`):

In [ ]:
def beste_naeherung(x, n):
    return kompromisse(x, 10 ** n - 1)[-1]


print('x          n=2: Bruch      Abstand     n=3: Bruch      Abstand     n=4: Bruch       Abstand')
for text in zahlen:
    x = Fraction(text)
    zeile = '%-10s' % text
    for n in (2, 3, 4):
        b = beste_naeherung(x, n)
        zeile = zeile + ' %9s %12.8f   ' % (b, float(abstand(x, b)))
    print(zeile)

x          n=2: Bruch      Abstand     n=3: Bruch      Abstand     n=4: Bruch       Abstand
1.857143        13/7   0.00000014         13/7   0.00000014         13/7   0.00000014   
1.293982       22/17   0.00013565      559/432   0.00000052    11783/9106   0.00000001   
3.141593      311/99   0.00017886      355/113   0.00000008      355/113   0.00000008   
1.414214       99/70   0.00007171     1393/985   0.00000080    12397/8766   0.00000001   


Das stimmt mit **Tabelle 5** der Lösungshinweise (Kettenbruch-Verfahren) überein. In Tabelle 4 (Stern-Brocot-Baum) steht bei n = 4 für 1,293982 der Bruch 5053/3905 und für 1,414214 der Bruch 1970/1393. Beide stehen auch auf unserer Kurve, werden aber von einem späteren Bruch mit ebenfalls vierstelligem Nenner übertroffen (11783/9106 bzw. 12397/8766). Das haben wir nachgerechnet, und wir halten uns an Tabelle 5.

Die Tabelle zeigt, was es heißt, die Ziele zu vereinbaren: Mehr Stellen helfen **nicht immer**. 1,857143 braucht nie mehr als 13/7, bei 3,141593 bringt die vierte Stelle gegenüber 355/113 nichts. Bei 1,293982 dagegen springt der Abstand von 0,00014 (n = 2) über 0,0000005 (n = 3) auf 0,00000001 (n = 4). Die Frage ist, **ab wann es sich nicht mehr lohnt**.

## Schritt 4: Die Rundung der Eingabe als Maßstab

Der Taschenrechner zeigt 1,293982 und meint eine Zahl zwischen 1,2939815 und 1,2939825. Alle Brüche in diesem Bereich sind **gleich gute** Antworten, weil wir die Zahl ohnehin nicht genauer kennen. Unser Maßstab ist deshalb:

> Nimm den Bruch mit dem **kleinsten Nenner**, der höchstens eine halbe Einheit der letzten Stelle von `x` entfernt ist.

Die Einheit der letzten Stelle erkennen wir an der Anzahl der Nachkommastellen im Eingabetext: bei `1.293982` sind es 6, die Toleranz ist `1 / (2 · 10^6)`. Die Schleife endet sicher, denn spätestens bei `q = 10^6` ist `x` selbst ein Bruch mit Abstand 0.

In [ ]:
def toleranz(text):
    nachkommastellen = len(text.split('.')[1])
    return Fraction(1, 2 * 10 ** nachkommastellen)


def einfachster(x, tol):
    q = 1
    while True:
        p = max(1, round(x * q))
        if abstand(x, Fraction(p, q)) <= tol:
            return Fraction(p, q)
        q = q + 1


for text in zahlen:
    x = Fraction(text)
    b = einfachster(x, toleranz(text))
    print(text, '->', b, ' Abstand', '%.8f' % float(abstand(x, b)), ' Toleranz', float(toleranz(text)))

1.857143 -> 13/7  Abstand 0.00000014  Toleranz 5e-07
1.293982 -> 3376/2609  Abstand 0.00000037  Toleranz 5e-07
3.141593 -> 355/113  Abstand 0.00000008  Toleranz 5e-07
1.414214 -> 1970/1393  Abstand 0.00000007  Toleranz 5e-07


Das ist unsere Antwort für die drei Zahlen der Aufgabe (und für die Zahl aus der Einleitung):

| Zahl | Ergebnis | Abstand |
|---|---|---|
| 1,857143 | 13/7 | 0,00000014 |
| 1,293982 | 3376/2609 | 0,00000037 |
| 3,141593 | 355/113 | 0,00000008 |
| 1,414214 | 1970/1393 | 0,00000007 |

**Vergleich mit den Tabellen der Hinweise.** Bei 1,857143 und 3,141593 kommt das Gleiche heraus wie bei den guten Verfahren dort (13/7, 355/113). Bei 1,414214 ist 1970/1393 der erste Bruch, der unter die Toleranz kommt (1393/985 hat Abstand 0,0000008, ist also für die sechste Stelle noch zu ungenau). Am wenigsten befriedigt 1,293982: Der Bruch 559/432 hat den Abstand 0,00000052, also **knapp über** der Toleranz von 0,0000005, und deshalb müssen wir bis zu 3376/2609 gehen. Das ist eine Schwäche jeder festen Grenze: Ein Bruch, der nur ein Haar daneben liegt, fliegt raus. Wer großzügiger ist und eine volle statt einer halben Einheit erlaubt, bekommt 559/432. Probiere es aus; die letzte Zeile zeigt, dass 0,333333 zu 1/3 und 0,142857 zu 1/7 wird, wie Annas Lehrer es erwartet:

In [ ]:
x = Fraction('1.293982')
print(einfachster(x, toleranz('1.293982')))
print(einfachster(x, 2 * toleranz('1.293982')))
print(einfachster(Fraction('0.333333'), toleranz('0.333333')), einfachster(Fraction('0.142857'), toleranz('0.142857')))

3376/2609
559/432
1/3 1/7


Zahlen größer als 1 lassen sich außerdem als gemischte Zahl schreiben, zum Beispiel 355/113 = 3 16/113:

In [ ]:
def gemischt(bruch):
    ganz = bruch.numerator // bruch.denominator
    rest = bruch - ganz
    if rest == 0:
        return str(ganz)
    if ganz == 0:
        return str(rest)
    return str(ganz) + ' ' + str(rest)


print(gemischt(Fraction(355, 113)), gemischt(Fraction(13, 7)), gemischt(Fraction(1, 3)), gemischt(Fraction(4, 2)))

3 16/113 1 6/7 1/3 2


## Das fertige Programm

Beide Verfahren zusammen: Mit `n` bekommt man die beste Näherung mit höchstens `n` Nennerstellen, ohne `n` den einfachsten Bruch innerhalb der Rundungsgenauigkeit.

In [ ]:
def rationalisiere(text, n=None):
    x = Fraction(text)
    if n is None:
        return einfachster(x, toleranz(text))
    return beste_naeherung(x, n)


for text in ['1.293982', '3.141593', '1.414214']:
    print(text, ' einfachster:', rationalisiere(text), '  höchstens 3 Stellen:', rationalisiere(text, 3))

1.293982  einfachster: 3376/2609   höchstens 3 Stellen: 559/432
3.141593  einfachster: 355/113   höchstens 3 Stellen: 355/113
1.414214  einfachster: 1970/1393   höchstens 3 Stellen: 1393/985


Rechnen mit Fließkommazahlen wäre hier möglich, aber `Fraction` ist sicherer: Bei den Vergleichen `abstand <= tol` zählt jede Stelle, und Fließkommazahlen würden Brüche, die genau an der Grenze liegen, zufällig auf die eine oder die andere Seite werfen.

## Zum Schluss: Geht das auch mit MiniZinc?

Ja, für das Verfahren „einfachster Bruch innerhalb der Rundungsgenauigkeit“: Gesucht sind zwei ganze Zahlen `p` und `q`, `q` möglichst klein, mit `|x − p/q| ≤ 1/(2·10^K)`. Weil `x = Z / 10^K` ein Bruch aus ganzen Zahlen ist, lässt sich das ohne Brüche schreiben: Multipliziert man die Ungleichung mit `2 · q · 10^K`, wird daraus `2 · |Z·q − p·10^K| ≤ q`. Das ist eine Bedingung mit `abs` auf ganzen Zahlen, und `minimize q` sucht den kleinsten Nenner. Der Zähler `p` ist dabei frei und wird von der Bedingung bestimmt. Die Suchstrategie `int_search([q, p], input_order, indomain_min)` probiert die Nenner von klein nach groß.

Die Daten kommen aus einer Python-Zelle: `Z` ist der Text ohne Punkt als ganze Zahl, `K` die Anzahl der Nachkommastellen.

```minizinc
int: Z;                       % x = Z / 10^K
int: K;
int: N = pow(10, K);
int: QMAX = 100000;           % so grosse Nenner betrachten wir hoechstens

var 1..QMAX: q;
var 1..(Z * QMAX div N + 2): p;

% |Z/N - p/q| <= 1/(2N)   <=>   2 * |Z*q - p*N| <= q
constraint 2 * abs(Z * q - p * N) <= q;

solve :: int_search([q, p], input_order, indomain_min) minimize q;
output ["p = \(p)\nq = \(q)\n"];
```

Die Datei `daten.dzn` für 1,293982:

```minizinc
Z = 1293982;
K = 6;
```

**Ausgabe** (CP-SAT):

```
p = 3376
q = 2609
```

**Messung bei allen vier Zahlen** (CP-SAT, jeweils sofort fertig):

| Zahl | p/q (MiniZinc) | Python |
|---|---|---|
| 1.857143 | 13/7 | 13/7 |
| 1.293982 | 3376/2609 | 3376/2609 |
| 3.141593 | 355/113 | 355/113 |
| 1.414214 | 1970/1393 | 1970/1393 |

MiniZinc findet in allen vier Fällen **genau die Brüche** aus Python, auch 3376/2609 für 1,293982.

**Grenzen:** Das Produkt `Z · q` wird bis zu etwa 3·10^10 groß. **Gecode und Chuffed** rechnen mit 32-Bit-Zahlen (bis etwa 2,1·10^9) und brechen schon beim Übersetzen mit der Meldung ab, die Zahl liege außerhalb des erlaubten Bereichs. Nur **CP-SAT** rechnet mit 64 Bit. Wir haben nur das Verfahren mit der Toleranz als Modell gebaut. Das Verfahren „beste Näherung mit höchstens `n` Stellen“ minimiert einen Quotienten, `|Z·q − p·10^K| / (10^K · q)`, und das passt schlecht zu ganzzahligen Bedingungen. Dafür ist die kurze Python-Schleife die bessere Wahl; MiniZinc zeigt hier, wie man „kleinster Nenner mit einer Genauigkeitsgrenze“ in einer einzigen Bedingung hinschreibt.

## Für Neugierige: Stern-Brocot-Baum und Kettenbrüche

Unsere Schleife probiert jeden Nenner. Bei einer Eingabe mit 6 Stellen ist das noch schnell, aber es geht klüger, und die Hinweise zeigen zwei Wege.

**Stern-Brocot-Baum.** Man hält zwei Brüche links und rechts von `x` fest, am Anfang `0/1` und `1/0`. Ihr **Mediant** (Zähler und Nenner einzeln addieren) liegt immer dazwischen. Liegt er unter dem erlaubten Bereich `[x − tol, x + tol]`, wird er der neue linke Rand; liegt er darüber, der neue rechte; liegt er im Bereich, sind wir fertig. Dieser erste Treffer ist der einfachste Bruch im Bereich.

In [ ]:
def einfachster_sb(x, tol):
    links = (0, 1)
    rechts = (1, 0)
    schritte = 0
    while True:
        mediant = (links[0] + rechts[0], links[1] + rechts[1])
        schritte = schritte + 1
        wert = Fraction(mediant[0], mediant[1])
        if wert < x - tol:
            links = mediant
        elif wert > x + tol:
            rechts = mediant
        else:
            return wert, schritte


for text in zahlen:
    wert, schritte = einfachster_sb(Fraction(text), toleranz(text))
    print(text, '->', wert, ' Schritte:', schritte, ' gleich wie vorher:', wert == einfachster(Fraction(text), toleranz(text)))

1.857143 -> 13/7  Schritte: 8  gleich wie vorher: True
1.293982 -> 3376/2609  Schritte: 39  gleich wie vorher: True
3.141593 -> 355/113  Schritte: 26  gleich wie vorher: True
1.414214 -> 1970/1393  Schritte: 18  gleich wie vorher: True


Bei den vier Zahlen braucht der Baum zwischen 8 und 39 Schritte; die Schleife über die Nenner brauchte bis zu 2609. Wir haben beide Funktionen an 300 zufälligen Zahlen mit 2 bis 5 Nachkommastellen verglichen, sie lieferten immer denselben Bruch.

Eine Besonderheit, die die Hinweise erwähnen: Beim Abstieg im Baum wird die Näherung **nicht** mit jedem Schritt besser; will man die beste Näherung mit begrenztem Nenner, muss man den ganzen Pfad betrachten und nicht nur das letzte Blatt. Für die Suche „erster Bruch im Bereich“ ist das hier unkritisch.

**Kettenbrüche.** Wendet man den euklidischen Algorithmus auf `x = a/b` an, entsteht die Folge der Quotienten `[a0; a1, a2, …]` mit `x = a0 + 1/(a1 + 1/(a2 + …))`. Schneidet man die Folge früher ab, erhält man die **Konvergenten**; sie sind besonders gute Näherungen, und die Hinweise suchen unter ihnen die beste für jede Stellenzahl. Die Folgen der vier Zahlen stimmen mit den Hinweisen überein:

In [ ]:
def kettenbruch(x):
    folge = []
    zaehler = x.numerator
    nenner = x.denominator
    while nenner != 0:
        folge.append(zaehler // nenner)
        zaehler, nenner = nenner, zaehler % nenner
    return folge


def konvergenten(folge):
    ergebnis = []
    for ende in range(1, len(folge) + 1):
        bruch = Fraction(folge[ende - 1])
        for a in reversed(folge[:ende - 1]):
            bruch = a + 1 / bruch
        ergebnis.append(bruch)
    return ergebnis


for text in zahlen:
    print(text, kettenbruch(Fraction(text)))
folge = kettenbruch(Fraction('1.414214'))
print([str(b) for b in konvergenten(folge)][:8])

1.857143 [1, 1, 6, 142857]
1.293982 [1, 3, 2, 2, 25, 10, 3, 2, 1, 1, 6]
3.141593 [3, 7, 16, 983, 4, 2]
1.414214 [1, 2, 2, 2, 2, 2, 2, 2, 3, 6, 1, 2, 1, 12]
['1', '3/2', '7/5', '17/12', '41/29', '99/70', '239/169', '577/408']


Zum Beispiel ist `1,857143 = [1; 1, 6, 142857]`; lässt man die große letzte Zahl weg, erhält man [1; 1, 6] = 13/7. Auffällig: 1,414214 ist fast √2 = [1; 2, 2, 2, …], deshalb die vielen Zweien. Die meisten besten Näherungen aus unserer Tabelle sind Konvergenten (13/7, 22/17, 559/432, 355/113, 99/70, 12397/8766). Drei sind es **nicht**: 311/99 für 3,141593 (n = 2), 1393/985 für 1,414214 (n = 3) und 11783/9106 für 1,293982 (n = 4). Es sind Zwischenbrüche, bei denen man ein Vielfaches des vorletzten Gliedes dazuzählt. Die Konvergenten allein reichen also für „die beste Näherung mit höchstens `n` Stellen“ nicht ganz; das haben wir nachgerechnet. Ein Kettenbruch hat nur so viele Glieder, wie der euklidische Algorithmus Schritte braucht, bei 6-stelligen Zahlen höchstens etwa 30. Wer also viele Zahlen verarbeiten muss, nimmt dieses Verfahren.

**Weiterdenken:**
- Die Hinweise erwähnen, dass man auch **glatte** Brüche bevorzugen könnte: x = 0,00501 eher als 1/200 (Abstand 0,00001) denn als 5/998 mit viel kleinerem Abstand. Wie könnte man eine 1 im Zähler oder Nullen am Ende der Zahlen bei der Bewertung belohnen?
- Wir haben den **absoluten** Abstand verwendet. Man kann auch den **relativen** Abstand `|x − p/q| / x` nehmen, für sehr kleine Zahlen wie 0,00501 ist das sinnvoller. Was ändert sich in `einfachster`, wenn die Toleranz mit `x` mitwächst?
- Schau dir die Kompromisse zu `2.718282` (Eulersche Zahl) oder `0.618034` (goldener Schnitt) an. Beim goldenen Schnitt sind die Konvergenten Quotienten benachbarter Fibonacci-Zahlen; erkennst du das an `kompromisse`?